In [1]:
from torch.utils.data import Dataset
from tqdm import tqdm
import json
import torch
from transformers import AutoTokenizer, DataCollatorForSeq2Seq
import re

class QwenDataset(Dataset):
    def __init__(self, model, data, system_instruction=None, max_len=512, inference_mode=False):
        self.inference_mode = inference_mode
        self.ignore_index = -100
        self.data = data
        self.max_len = max_len
        self.model = model
        self.tokenizer = AutoTokenizer.from_pretrained(
            model, 
            trust_remote_code=True, 
            padding_side='right', 
        )
        
        if not system_instruction:
            self.system_instruction = "Analyze the text. Rate Valence (positivity) and Arousal (intensity) on a scale of 1.00-9.00. Output format: Valence#Arousal."
        else:
            self.system_instruction = system_instruction

    def __len__(self):
        return len(self.data)

    @staticmethod
    def domain_retrieval(path_name: str):
        path_name = path_name.lower()
        if "restaraunt" in path_name or "restaurant" in path_name:
            return 'restaurant'
        elif "laptop" in path_name:
            return "laptop"
        elif "finance" in path_name:
            return "finance"
        else:
            return "general"

    @staticmethod
    def _parse_jsonl(path):
        flattened_data = []
        current_domain = QwenDataset.domain_retrieval(path)
        print("Parsing JSONL data from:", path)
        
        with open(path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            
        for line in tqdm(lines, desc="Loading Data"):
            if not line.strip(): continue
            entry = json.loads(line)
            
            if 'Quadruplet' in entry:
                for quad in entry['Quadruplet']:
                    aspect = quad.get('Aspect', 'NULL')
                    target = quad.get('Category', 'general').replace("#", " ") if aspect == "NULL" else aspect
                    try:
                        val, aro = map(float, quad.get('VA', '5.0#5.0').split('#'))
                    except:
                        val, aro = 5.0, 5.0
                    
                    flattened_data.append({
                        'Domain': current_domain, 'Text': entry.get('Text'), 'Target': str(target),
                        'Valence': val, 'Arousal': aro , 'Aspect' : aspect 
                    })
            else:
                flattened_data.append({
                    'Domain': current_domain, 'Text': entry.get('Text'), 'Target': "General",
                    'Valence': 5.0, 'Arousal': 5.0 , 'Aspect' : aspect 
                })
        return flattened_data 

    def __getitem__(self, idx):
        row = self.data[idx]
        
        system_content = self.system_instruction
        user_content = f"Domain:{row['Domain']}\nText:{row['Text']}\nTarget:{row['Aspect']}"
        assistant_content  = f"Valence:{row['Valence']}Arousal:{row['Arousal']}"

        message = [
            {"role" :  "system" , "content" : system_content}, 
            {"role":   "user",    "content": user_content}, 
            {"role" :  "assistant" , "content" : assistant_content} , 
        ]
        prompt_message = [ 
                {"role" :  "system" , "content" : system_content}, 
                {"role":   "user",    "content": user_content}
        ]

        if self.inference_mode: 
            prompt = self.tokenizer.apply_chat_template(
                prompt_message,
                add_generation_prompt=True,
                tokenize=False,
                enable_thinking=False
            )

            tokenized_prompt = self.tokenizer(
                prompt,
                truncation=True,
                max_length=self.max_len,
                padding=False, 
                return_tensors="pt"
            )

            return {
                "input_ids" :tokenized_prompt['input_ids'].squeeze(0), 
                "attention_mask" : tokenized_prompt['attention_mask'].squeeze(0)
            }

        full_message_chat = self.tokenizer.apply_chat_template(
            message ,
            enable_thinking=False, 
            add_generation_prompt=False, 
            tokenize=False
        )
        # Cleanup
        full_message_chat = full_message_chat.replace("<think>\n\n</think>", "").replace("<think>\n</think>", "").replace("<think></think>", "")
        
        tokenized_full = self.tokenizer(
            full_message_chat, 
            truncation=True, 
            max_length=self.max_len, 
            padding=False,
            return_tensors='pt'
        )
        input_ids = tokenized_full['input_ids'].squeeze(0)
        attention_mask = tokenized_full['attention_mask'].squeeze(0)

        labels = input_ids.clone()
        
        prompt_message_chat = self.tokenizer.apply_chat_template(
            prompt_message, 
            add_generation_prompt=True, 
            tokenize=False
        )
        prompt_message_chat = prompt_message_chat.replace("<think>", "").replace("</think>", "")
        prompt_message_chat = re.sub(r"(<\|im_start\|>assistant)\s+", r"\1\n", prompt_message_chat)

        tokenized_prompt = self.tokenizer(
            prompt_message_chat,
            truncation=True, 
            max_length=self.max_len, 
            padding=False, 
            return_tensors='pt'
        )
        prompt_ids = tokenized_prompt['input_ids'].squeeze(0)

        mask_len = 0
        min_len = min(len(prompt_ids), len(input_ids))
        
        for i in range(min_len):
            if prompt_ids[i] == input_ids[i]:
                mask_len = i + 1
            else:
                break
        

        labels[:mask_len] = -100

        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

KeyboardInterrupt: 

In [ ]:
from torch.utils.data import Dataset
from tqdm import tqdm
import json
import torch
from transformers import AutoTokenizer, DataCollatorForSeq2Seq
import re

class QwenDataset(Dataset):
    def __init__(self, model, data, system_instruction=None, max_len=512, inference_mode=False):
        self.inference_mode = inference_mode
        self.ignore_index = -100
        self.data = data
        self.max_len = max_len
        self.model = model
        self.tokenizer = AutoTokenizer.from_pretrained(
            model, 
            trust_remote_code=True, 
            padding_side='right', 
        )
        
        if not system_instruction:
            self.system_instruction = "Analyze the text. Rate Valence (positivity) and Arousal (intensity) on a scale of 1.00-9.00. Output format: Valence#Arousal."
        else:
            self.system_instruction = system_instruction

    def __len__(self):
        return len(self.data)

    @staticmethod
    def domain_retrieval(path_name: str):
        path_name = path_name.lower()
        if "restaraunt" in path_name or "restaurant" in path_name:
            return 'restaurant'
        elif "laptop" in path_name:
            return "laptop"
        elif "finance" in path_name:
            return "finance"
        else:
            return "general"

    @staticmethod
    def _parse_jsonl(path):
        flattened_data = []
        current_domain = QwenDataset.domain_retrieval(path)
        print("Parsing JSONL data from:", path)
        
        with open(path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            
        for line in tqdm(lines, desc="Loading Data"):
            if not line.strip(): continue
            entry = json.loads(line)
            
            if 'Quadruplet' in entry:
                for quad in entry['Quadruplet']:
                    aspect = quad.get('Aspect', 'NULL')
                    target = quad.get('Category', 'general').replace("#", " ") if aspect == "NULL" else aspect
                    try:
                        val, aro = map(float, quad.get('VA', '5.0#5.0').split('#'))
                    except:
                        val, aro = 5.0, 5.0
                    
                    flattened_data.append({
                        'Domain': current_domain, 'Text': entry.get('Text'), 'Target': str(target),
                        'Valence': val, 'Arousal': aro , 'Aspect' : aspect 
                    })
            else:
                flattened_data.append({
                    'Domain': current_domain, 'Text': entry.get('Text'), 'Target': "General",
                    'Valence': 5.0, 'Arousal': 5.0 , 'Aspect' : aspect 
                })
        return flattened_data 

    def __getitem__(self, idx):
        row = self.data[idx]
        
        system_content = self.system_instruction
        user_content = f"Domain:{row['Domain']}\nText:{row['Text']}\nTarget:{row['Aspect']}"
        assistant_content  = f"Valence:{row['Valence']}Arousal:{row['Arousal']}"

        message = [
            {"role": "system", "content": system_content}, 
            {"role": "user",   "content": user_content}, 
            {"role": "assistant", "content": assistant_content} 
        ]
        
        prompt_message = [ 
                {"role": "system", "content": system_content}, 
                {"role": "user",   "content": user_content}
        ]

        if self.inference_mode: 
            prompt = self.tokenizer.apply_chat_template(
                prompt_message,
                add_generation_prompt=True,
                tokenize=False,
                enable_thinking=False
            )

            tokenized_prompt = self.tokenizer(
                prompt,
                truncation=True,
                max_length=self.max_len,
                padding=False, 
                return_tensors="pt"
            )

            return {
                "input_ids": tokenized_prompt['input_ids'].squeeze(0), 
                "attention_mask": tokenized_prompt['attention_mask'].squeeze(0)
            }

        full_message_chat = self.tokenizer.apply_chat_template(
            message,
            enable_thinking=False, 
            add_generation_prompt=False, 
            tokenize=False
        )
        
        full_message_chat = full_message_chat.replace("<think>\n\n</think>", "").replace("<think>\n</think>", "").replace("<think></think>", "")
        
        
        tokenized_full = self.tokenizer(
            full_message_chat, 
            truncation=True, 
            max_length=self.max_len, 
            padding=False,
            return_tensors='pt'
        )

        input_ids = tokenized_full['input_ids'].squeeze(0)
        attention_mask = tokenized_full['attention_mask'].squeeze(0)
        labels = input_ids.clone()


        prompt_message_chat = self.tokenizer.apply_chat_template(
            prompt_message, 
            enable_thinking=False, 
            add_generation_prompt=True, 
            tokenize=False
        )

        tokenized_prompt = self.tokenizer(
            prompt_message_chat,
            truncation=True, 
            max_length=self.max_len, 
            padding=False, 
            return_tensors='pt'
        )
        

        input_ids_prompt_len = len(tokenized_prompt['input_ids'].squeeze(0))

        if input_ids_prompt_len >= len(labels):
            input_ids_prompt_len = len(labels) - 1

        labels[:input_ids_prompt_len] = -100

        return {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels
            }


In [ ]:
import torch 
import math
import numpy as np
import os 
import pandas as pd
import random
from transformers import TrainerCallback

class SpaceSaverCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if not state.best_model_checkpoint:
            return
        checkpoint_dir = state.best_model_checkpoint
        optim_file = os.path.join(checkpoint_dir, "optimizer.pt")
        sched_file = os.path.join(checkpoint_dir, "scheduler.pt")
        try:
            if os.path.exists(optim_file):
                os.remove(optim_file)
                print(f"SpaceSaver: Deleted {optim_file}")
            if os.path.exists(sched_file):
                os.remove(sched_file)
        except Exception as e:
            print(f"Could not clean up checkpoint: {e}")

class PrinterCallback(TrainerCallback):
    def __init__(self, tokenizer, model, prompt_text):
        self.tokenizer = tokenizer
        self.model = model
        self.prompt_text = prompt_text

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % 50 == 0:
            print(f"\nGENERATION CHECK AT STEP {state.global_step} ---")        
    
            inputs = self.tokenizer(self.prompt_text, return_tensors="pt").to(self.model.device)
            
            self.model.eval()
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs, 
                    max_new_tokens=64, 
                    do_sample=True, 
                    temperature=0.7
                )
            self.model.train() 
            
            new_tokens = outputs[0][inputs.input_ids.shape[1]:]
            generated_text = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
            print(f"INPUT: ...{self.prompt_text[-50:].strip()}")
            print(f"OUTPUT: {generated_text.strip()}")
            print("--------------------------------------------------\n")

class Cherrypiocker(TrainerCallback):
    def __init__(self, tokenizer, eval_dataset, num_samples=3):
        self.tokenizer = tokenizer
        self.eval_dataset = eval_dataset
        self.num_samples = num_samples

    def on_evaluate(self, args, state, control, model, **kwargs):
        print(f"\nGenerating Samples at Step {state.global_step}")
        
        indices = random.sample(range(len(self.eval_dataset)), self.num_samples)
        model.eval()
        
        for idx in indices:
            item = self.eval_dataset[idx]
            
            
            input_ids_full = item['input_ids']
            labels = item['labels']
            
            
            if not isinstance(labels, torch.Tensor):
                 labels = torch.tensor(labels)
            
            answer_starts = (labels != -100).nonzero(as_tuple=True)[0]
            
            if len(answer_starts) == 0:
                answer_start_idx = len(input_ids_full) // 2
            else:
                answer_start_idx = answer_starts[0].item()

            
            input_ids = item['input_ids'][:answer_start_idx].unsqueeze(0).to(model.device)
            attention_mask = item['attention_mask'][:answer_start_idx].unsqueeze(0).to(model.device)

            with torch.no_grad():
                output_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=64,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    temperature = 0.7,
                    do_sample=False
                )
            
            prompt_text = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
            
            new_tokens = output_ids[0][input_ids.shape[1]:]
            generated_text = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
            
            
            gold_ids = input_ids_full[answer_start_idx:]
            gold_text = self.tokenizer.decode(gold_ids, skip_special_tokens=True)

            print("-" * 50)
            print(f"Input: ...{prompt_text[-100:].replace(chr(10), ' ')}")
            print(f"Gold:   {gold_text}")
            print(f"Model:  {generated_text}")
            print("-" * 50)
            
        model.train()

In [ ]:
import torch
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("CRITICAL WARNING: CUDA not available. Training will run on CPU and take a VERY long time.")

PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA device count: 2
Current device: 0
Device name: Tesla T4


In [ ]:
# # Push model to a private Hugging Face repo (set these before running)
# HF_TOKEN = ""
# REPO_ID = "ssurface/mmbert-cls-32-bin-alldata"
# LOCAL_MODEL_DIR = "/kaggle/working/models/mmBERT-base-finetuned-20260210_111301/final"

# from huggingface_hub import login
# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# if not HF_TOKEN:
#     raise ValueError("Set HF_TOKEN before running.")

# login(token=HF_TOKEN)

# model = AutoModelForSequenceClassification.from_pretrained(LOCAL_MODEL_DIR, trust_remote_code=True)
# # tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR, trust_remote_code=True)

# model.push_to_hub(REPO_ID, private=True)
# # tokenizer.push_to_hub(REPO_ID, private=True)

# print(f"Pushed model and tokenizer to {REPO_ID} (private)")

In [ ]:
# Parameters (edit these)
MODEL_NAME = "Qwen/Qwen3-0.6B"
TRAIN_DATA_PATH = "/kaggle/working/dimabsa/out_put_plit_no_emobank/train.jsonl"
EVAL_DATA_PATH = "/kaggle/working/dimabsa/out_put_plit_no_emobank/dev.jsonl"
TEST_DATA_PATH = "/kaggle/working/dimabsa/out_put_plit_no_emobank/test.jsonl"
OUTPUT_DIR = "/kaggle/working/dimabsa/qwen3/logs_qwen"

EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 2
LR = 1e-5

WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
EVAL_STRATEGY = "steps"
EVAL_STEPS = 10
SAVE_STRATEGY = "steps"
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2
REPORT_TO = "none"
DATALOADER_NUM_WORKERS = 2
REMOVE_UNUSED_COLUMNS = False
GROUP_BY_LENGTH = True
DDP_FIND_UNUSED_PARAMETERS = False

USE_BF16 = None  # Set True/False, or leave None to auto-detect
USE_FP16 = None  # Set True/False, or leave None to auto-detect
DEVICE_MAP = "cuda"
PAD_TO_MULTIPLE_OF = 8

In [ ]:
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

print(f"Loading Model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

if USE_BF16 is None and USE_FP16 is None:
    use_bf16 = torch.cuda.is_bf16_supported()
    use_fp16 = not use_bf16
else:
    use_bf16 = bool(USE_BF16)
    use_fp16 = bool(USE_FP16)

dtype = torch.bfloat16 if use_bf16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=dtype,
    device_map=DEVICE_MAP,
    # use_cache=False,
)

print("Loading and Parsing Datasets...")
train_raw = QwenDataset._parse_jsonl(TRAIN_DATA_PATH)
eval_raw = QwenDataset._parse_jsonl(EVAL_DATA_PATH)
# test_raw = QwenDataset._parse_jsonl(TEST_DATA_PATH)

train_dataset = QwenDataset(MODEL_NAME, train_raw)
eval_dataset = QwenDataset(MODEL_NAME, eval_raw)

sample = train_dataset[0]
print(f"\n[DEBUG] Sample Input Shape: {sample['input_ids'].shape}")
print("[DEBUG] Decoded Labels (Masked parts hidden):")
valid_labels = sample["labels"][sample["labels"] != -100]
print(tokenizer.decode(valid_labels))
print("----------------------------------------------------\n")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    eval_strategy=EVAL_STRATEGY,
    eval_steps=EVAL_STEPS,
    save_strategy=SAVE_STRATEGY,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    bf16=use_bf16,
    fp16=use_fp16,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    report_to=REPORT_TO,
    remove_unused_columns=REMOVE_UNUSED_COLUMNS,
    group_by_length=GROUP_BY_LENGTH,
    ddp_find_unused_parameters=DDP_FIND_UNUSED_PARAMETERS,
)

sample_prompt = "Domain:restaurant\nText:The food was amazing but the service was slow.\nTarget:service"
sample_msg = [
    {
        "role": "system",
        "content": "Analyze the text. Rate Valence (positivity) and Arousal (intensity) on a scale of 1.00-9.00. Output format: Valence#Arousal.",
    },
    {"role": "user", "content": sample_prompt},
]
formatted_prompt = tokenizer.apply_chat_template(
    sample_msg,
    add_generation_prompt=True,
    tokenize=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer,
        pad_to_multiple_of=PAD_TO_MULTIPLE_OF,
        return_tensors="pt",
        padding=True,
    ),
    callbacks=[
        SpaceSaverCallback(),
        Cherrypiocker(tokenizer, eval_dataset, num_samples=2),
        PrinterCallback(tokenizer, model, formatted_prompt),
    ],
)

print("Starting Training...")
trainer.train()

print(f"Saving model to {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Loading Model: Qwen/Qwen3-0.6B


Loading and Parsing Datasets...
Parsing JSONL data from: /kaggle/working/dimabsa/out_put_plit_no_emobank/train.jsonl


Loading Data: 100%|██████████| 18594/18594 [00:00<00:00, 147101.11it/s]


Parsing JSONL data from: /kaggle/working/dimabsa/out_put_plit_no_emobank/dev.jsonl


Loading Data: 100%|██████████| 2323/2323 [00:00<00:00, 133191.64it/s]



[DEBUG] Sample Input Shape: torch.Size([117])
[DEBUG] Decoded Labels (Masked parts hidden):



Valence:6.62Arousal:5.75<|im_end|>

----------------------------------------------------

Starting Training...


KeyboardInterrupt: 

In [ ]:
# import os
# import torch
# from transformers import (
#     AutoModelForCausalLM,
#     AutoTokenizer,
#     TrainingArguments,
#     Trainer,
#     DataCollatorForSeq2Seq,
# )
# from peft import LoraConfig, get_peft_model, TaskType

# print(f"Loading Model for LoRA: {MODEL_NAME}")

# # Initialize model with LoRA adapters
# model_lora = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     trust_remote_code=True,
#     torch_dtype=dtype,
#     device_map=DEVICE_MAP,
# )

# # LoRA Configuration
# # We target all linear layers for maximum effectiveness in adaptation
# lora_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["q_proj", "k_proj", "v_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type=TaskType.CAUSAL_LM
# )

# # Wrap model with PEFT
# model_lora = get_peft_model(model_lora, lora_config)
# model_lora.print_trainable_parameters()

# # Define a separate output directory for LoRA experiment
# lora_output_dir = os.path.join(os.path.dirname(OUTPUT_DIR), "qwen3_lora")

# training_args_lora = TrainingArguments(
#     output_dir=lora_output_dir,
#     num_train_epochs=EPOCHS,
#     per_device_train_batch_size=BATCH_SIZE,
#     per_device_eval_batch_size=BATCH_SIZE,
#     gradient_accumulation_steps=GRAD_ACCUM,
#     learning_rate=LR * 2, # LoRA often benefits from a slightly higher LR than full fine-tuning
#     weight_decay=WEIGHT_DECAY,
#     logging_steps=LOGGING_STEPS,
#     eval_strategy=EVAL_STRATEGY,
#     eval_steps=EVAL_STEPS,
#     save_strategy=SAVE_STRATEGY,
#     save_steps=SAVE_STEPS,
#     save_total_limit=SAVE_TOTAL_LIMIT,
#     bf16=use_bf16,
#     fp16=use_fp16,
#     dataloader_num_workers=DATALOADER_NUM_WORKERS,
#     report_to=REPORT_TO,
#     remove_unused_columns=REMOVE_UNUSED_COLUMNS,
#     group_by_length=GROUP_BY_LENGTH,
#     ddp_find_unused_parameters=DDP_FIND_UNUSED_PARAMETERS,
# )

# trainer_lora = Trainer(
#     model=model_lora,
#     args=training_args_lora,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     data_collator=DataCollatorForSeq2Seq(
#         tokenizer,
#         pad_to_multiple_of=PAD_TO_MULTIPLE_OF,
#         return_tensors="pt",
#         padding=True,
#     ),
#     callbacks=[
#         SpaceSaverCallback(),
#         Cherrypiocker(tokenizer, eval_dataset, num_samples=2),
#         PrinterCallback(tokenizer, model_lora, formatted_prompt),
#     ],
# )

# print("Starting LoRA Training...")
# trainer_lora.train()

# print(f"Saving LoRA model to {lora_output_dir}")
# trainer_lora.save_model(lora_output_dir)
# tokenizer.save_pretrained(lora_output_dir)

Loading Model for LoRA: Qwen/Qwen3-0.6B


trainable params: 3,211,264 || all params: 599,261,184 || trainable%: 0.5359
Starting LoRA Training...


NameError: name 'system_instruction' is not defined